# Model Training v3 — Speaker-Level Grouped Pipeline

This v3 notebook implements the higher-impact changes from v2 diagnostics:
- **Speaker-level aggregation** across all available tokens
- **Multi-token setup** (`SELECTED_TOKEN=None`) to reduce utterance noise
- **Grouped cross-validation model search** with `StratifiedGroupKFold`
- **Feature-size search** (`SelectKBest(k)`) instead of fixed `k=50`
- **Binary threshold tuning** for better macro-F1 / balanced accuracy

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif, SelectFromModel, RFE
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sys.path.append('..')
from src.features import FeatureOptions, load_feature_tables

In [ ]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Keep all tokens, then aggregate per speaker
SELECTED_TOKEN = None

# Class filtering (applied after grouping)
MIN_SAMPLES_PER_CLASS = 50
MAX_SAMPLES_PER_CLASS = 500

# Healthy handling:
# - balance_healthy_to_pathological: downsample healthy if it exceeds pathological total
# - upsample_healthy_to_pathological: DO NOT duplicate rows; simply skip healthy class cap
BALANCE_HEALTHY_TO_PATHOLOGICAL = True
UPSAMPLE_HEALTHY_TO_PATHOLOGICAL = True

# Grouped CV + feature size search
N_SPLITS = 5
K_CANDIDATES = [15, 20, 30, 50, 80]
THRESHOLD_GRID = np.linspace(0.30, 0.70, 21)

TARGET_SOURCE_COL_PREFERENCE = 'pathology_de'
USE_GROUPED_TARGET = True
KEEP_UNMAPPED_LABELS = True
UNMAPPED_LABEL = 'Other'

DISEASE_GROUP_MAP = {
    # --- NEUROLOGICAL ---
    'Morbus Parkinson': 'Neurological',
    'Rekurrensparese': 'Neurological',
    'Spasmodische Dysphonie': 'Neurological',

    # --- STRUCTURAL ---
    'Phonationsknötchen': 'Structural',
    'Stimmlippenpolyp': 'Structural',
    'Reinke Ödem': 'Structural',
    'Laryngitis': 'Structural',

    # --- FUNCTIONAL ---
    'Hypotone Dysphonie': 'Functional',
    'Hyperfunktionelle Dysphonie': 'Functional',
    'Psychogene Dysphonie': 'Functional',
    'Funktionelle Dysphonie': 'Functional',

    # --- REDUNDANT ---
    'Phonationsknötchen': 'Structural',
    'Reinke Ödem': 'Structural',
}

opts = FeatureOptions(
    prefix=Path('..'),
    include_splits=True,
    random_seed=RANDOM_SEED,
    max_samples_per_class=MAX_SAMPLES_PER_CLASS,
    balance_healthy_to_pathological=BALANCE_HEALTHY_TO_PATHOLOGICAL,
    upsample_healthy_to_pathological=UPSAMPLE_HEALTHY_TO_PATHOLOGICAL,
    selected_token=SELECTED_TOKEN,
    num_workers=None,
)
opts

In [ ]:
tables = load_feature_tables(options=opts, build_if_missing=True, save_if_built=True)

for name, tdf in tables.items():
    print(f'{name}: {tdf.shape}')

core_df = tables['core'].copy()
acoustic_df = tables['acoustic'].copy()
multifractal_df = tables['multifractal'].copy()
opensmile_df = tables.get('opensmile', pd.DataFrame()).copy()
splits_df = tables.get('splits', pd.DataFrame()).copy()

df = core_df.merge(acoustic_df, on='sample_key', how='left')
df = df.merge(multifractal_df, on='sample_key', how='left')
if not opensmile_df.empty:
    df = df.merge(opensmile_df, on='sample_key', how='left')
if not splits_df.empty:
    df = df.merge(splits_df, on='sample_key', how='left')

if 'feature_status' in df.columns:
    df = df[df['feature_status'].isin(['ok', 'partial_failure'])].copy()
if 'acoustic_status' in df.columns:
    df = df[df['acoustic_status'] == 'ok'].copy()
if 'mf_status' in df.columns:
    df = df[df['mf_status'] == 'ok'].copy()
if 'opensmile_status' in df.columns:
    df = df[df['opensmile_status'] == 'ok'].copy()

print('Merged sample-level shape:', df.shape)
print('Unique speakers (sample level):', df['speaker_id'].nunique())

In [ ]:
# Build target labels
if TARGET_SOURCE_COL_PREFERENCE in df.columns:
    target_source_col = TARGET_SOURCE_COL_PREFERENCE
elif 'pathology_en' in df.columns:
    target_source_col = 'pathology_en'
elif 'pathology_de' in df.columns:
    target_source_col = 'pathology_de'
else:
    raise ValueError('Could not find pathology label column (expected pathology_de/pathology_en).')

raw_target = df[target_source_col].astype(str).str.strip()
if USE_GROUPED_TARGET:
    mapped_target = raw_target.map(DISEASE_GROUP_MAP)
    if KEEP_UNMAPPED_LABELS:
        mapped_target = mapped_target.fillna(raw_target)
    else:
        mapped_target = mapped_target.fillna(UNMAPPED_LABEL)
    df['target_label'] = mapped_target.astype(str)
else:
    df['target_label'] = raw_target

target_col = 'target_label'

print('Target source:', target_source_col)
print('Grouped target enabled:', USE_GROUPED_TARGET)
display(df[target_col].value_counts().to_frame('sample_count_before_filter'))

small_classes = df[target_col].value_counts()[df[target_col].value_counts() < MIN_SAMPLES_PER_CLASS].index.tolist()
if small_classes:
    print(f'Dropping classes with < {MIN_SAMPLES_PER_CLASS} samples: {small_classes}')
    df = df[~df[target_col].isin(small_classes)].copy()

display(df[target_col].value_counts().to_frame('sample_count_after_filter'))

In [ ]:
# Speaker-level aggregation (mean + std only — median adds noise with limited samples)
meta_cols = {
    'sample_key', 'duplicate_class_key', 'recording_id', 'speaker_id', 'wav_path',
    'feature_status', 'feature_error', 'acoustic_status', 'acoustic_error',
    'mf_status', 'mf_error', 'opensmile_status', 'opensmile_error',
    'split', 'split_seed', 'pathology_de', 'pathology_en', target_col, 'is_healthy',
}

numeric_feature_cols = [
    c for c in df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])
]

agg_stats = ['mean', 'std']
num_agg = df.groupby('speaker_id')[numeric_feature_cols].agg(agg_stats)
num_agg.columns = [f'{col}__{stat}' for col, stat in num_agg.columns]

speaker_meta = df.groupby('speaker_id').agg({
    'sex': 'first',
    'is_healthy': lambda s: int(round(float(s.mean()))),
    target_col: lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0],
    'pathology_de': lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0],
    'pathology_en': lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0],
    'recording_id': 'nunique',
    'token': 'nunique',
}).rename(columns={'recording_id': 'n_recordings', 'token': 'n_tokens'})

speaker_df = speaker_meta.join(num_agg, how='inner').reset_index()

# Replace NaN std for single-observation speakers
std_cols = [c for c in speaker_df.columns if c.endswith('__std')]
if std_cols:
    speaker_df[std_cols] = speaker_df[std_cols].fillna(0.0)

print('Speaker-level shape:', speaker_df.shape)
display(speaker_df[[target_col, 'is_healthy', 'n_tokens', 'n_recordings']].head())
display(speaker_df[target_col].value_counts().to_frame('speaker_count_by_class'))

In [ ]:
# Model-ready matrices (speaker level)
feature_exclude = {'speaker_id', 'is_healthy', target_col, 'pathology_de', 'pathology_en'}
speaker_numeric_cols = [
    c for c in speaker_df.columns if c not in feature_exclude and pd.api.types.is_numeric_dtype(speaker_df[c])
]
speaker_categorical_cols = ['sex'] if 'sex' in speaker_df.columns else []

X_spk = speaker_df[speaker_numeric_cols + speaker_categorical_cols].copy()
y_bin_spk = speaker_df['is_healthy'].astype(int).copy()
y_multi_spk = speaker_df[target_col].astype(str).copy()
groups_spk = speaker_df['speaker_id'].astype(str).copy()

print('Speakers:', len(speaker_df))
print('Binary classes:', y_bin_spk.value_counts().to_dict())
print('Multi classes:', y_multi_spk.value_counts().to_dict())
print('Numeric features:', len(speaker_numeric_cols))
print('Categorical features:', speaker_categorical_cols)

In [ ]:
# Advanced preprocessor: using PowerTransformer to normalize heavy-tailed features like MF/Jitter
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)),
    ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)), # Better than StandardScaler for acoustic data
    ('variance', VarianceThreshold(threshold=0.0)),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, speaker_numeric_cols),
        ('cat', categorical_transformer, speaker_categorical_cols),
    ],
    remainder='drop',
)

def make_pipe(model, sel_type='kbest', k=50, l1_c=0.05):
    """
    Flexible pipeline builder for extreme grid search.
    sel_type options:
    - 'kbest': standard f_classif
    - 'mi': mutual_info_classif
    - 'l1': LinearSVC-based selection
    - 'rfe': Recursive Feature Elimination (requires tree-based estimator)
    """
    if sel_type == 'l1':
        selector = SelectFromModel(
            LinearSVC(penalty="l1", dual=False, C=l1_c, random_state=RANDOM_SEED, class_weight='balanced', max_iter=2000)
        )
    elif sel_type == 'rfe':
        # RFE works backward, pruning weakest features using an internal model
        estimator = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
        selector = RFE(estimator, n_features_to_select=k, step=10)
    elif sel_type == 'mi':
        selector = SelectKBest(mutual_info_classif, k=k)
    else: # kbest f_classif
        selector = SelectKBest(f_classif, k=k)
        
    return Pipeline(steps=[
        ('prep', preprocessor),
        ('selector', selector),
        ('model', model),
    ])

In [ ]:
# Binary CV with per-fold threshold tuning (optimise balanced_accuracy)
def evaluate_binary_grouped_cv(model, sel_type, k, X, y, groups, n_splits=5, threshold_grid=None, l1_c=0.05):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    threshold_grid = THRESHOLD_GRID if threshold_grid is None else threshold_grid

    fold_rows = []
    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        pipe = make_pipe(clone(model), sel_type=sel_type, k=k, l1_c=l1_c)
        pipe.fit(X_tr, y_tr)

        # Tune threshold on train probabilities (handle hard voting which lacks predict_proba)
        if hasattr(pipe.named_steps['model'], "predict_proba"):
            p_tr = pipe.predict_proba(X_tr)[:, 1]
            best_thr, best_ba = 0.5, -1.0
            for thr in threshold_grid:
                pred_tr = (p_tr >= thr).astype(int)
                ba = balanced_accuracy_score(y_tr, pred_tr)
                if ba > best_ba:
                    best_ba = ba
                    best_thr = float(thr)

            p_te = pipe.predict_proba(X_te)[:, 1]
            pred_te = (p_te >= best_thr).astype(int)
        else:
            # Hard voting fallback (no threshold tuning possible)
            best_thr = 0.5
            pred_te = pipe.predict(X_te)

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(y_te, pred_te),
            'balanced_accuracy': balanced_accuracy_score(y_te, pred_te),
            'f1_macro': f1_score(y_te, pred_te, average='macro', zero_division=0),
        })

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'accuracy_std': fold_df['accuracy'].std(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'balanced_accuracy_std': fold_df['balanced_accuracy'].std(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
        'f1_macro_std': fold_df['f1_macro'].std(),
        'threshold_mean': fold_df['threshold'].mean(),
        'threshold_std': fold_df['threshold'].std(),
    }
    return fold_df, summary

In [ ]:
# Robust Stacking/Voting Classifiers on top of heavily tuned base estimators
rf_base = RandomForestClassifier(
    n_estimators=1200, max_depth=16, min_samples_leaf=3,
    class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1
)
xgb_base = XGBClassifier(
    n_estimators=1000, learning_rate=0.015, max_depth=5,
    subsample=0.75, colsample_bytree=0.5, min_child_weight=4,
    reg_alpha=3.0, reg_lambda=6.0,
    random_state=RANDOM_SEED, n_jobs=-1, eval_metric='logloss'
)
lgb_base = LGBMClassifier(
    n_estimators=1000, learning_rate=0.015, num_leaves=24,
    min_child_samples=6, reg_alpha=3.0, reg_lambda=6.0,
    subsample=0.75, colsample_bytree=0.5,
    class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
)

# estimators = [
#     ('rf', clone(rf_base)),
#     ('xgb', clone(xgb_base)),
#     ('lgb', clone(lgb_base))
# ]

binary_models = {
    'XGBoost': xgb_base,
    'RandomForest': rf_base,
    # 'Voting-Hard': VotingClassifier(estimators=estimators, voting='hard'),
    # 'Voting-Soft': VotingClassifier(estimators=estimators, voting='soft'),
    # Stacking learns a meta-model (LogisticRegression) over the underlying models
    # 'Stacking': StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(class_weight='balanced'), cv=3, n_jobs=-1),
}

bin_rows = []
# Comprehensive advanced selection: We know L1 works best, so we hyperfocus there, but add RFE.
experiments = [
    {'sel': 'l1', 'k': 'auto', 'c': 0.05},
    {'sel': 'l1', 'k': 'auto-loose', 'c': 0.1},
    {'sel': 'rfe', 'k': 50, 'c': None},
    {'sel': 'rfe', 'k': 80, 'c': None},
]

for exp in experiments:
    for model_name, model in binary_models.items():
        # Stacking lacks predict_proba natively if nested oddly with thresholds, so for Hard voting we skip threshold tuning
        if model_name in ['Voting-Hard']:
            pass_grid = [0.5] # Hard voting outputs strict 0/1
        else:
            pass_grid = THRESHOLD_GRID
            
        # RFE passes an int K; L1 ignores the exact K inside the pipeline
        pass_k = exp['k'] if isinstance(exp['k'], int) else 50 
        
        fold_df, summ = evaluate_binary_grouped_cv(
            model,
            sel_type=exp['sel'],
            k=pass_k,
            X=X_spk, y=y_bin_spk, groups=groups_spk, 
            n_splits=N_SPLITS, 
            threshold_grid=pass_grid,
            l1_c=exp['c'] if exp['c'] is not None else 0.05
        )
        label = f"{exp['k']} ({str(exp['sel']).upper()})"
        bin_rows.append({
            'model': model_name,
            'k': label,
            **summ,
        })

bin_results = pd.DataFrame(bin_rows).sort_values(
    by=['balanced_accuracy_mean', 'f1_macro_mean'], ascending=False
).reset_index(drop=True)

display(bin_results.head(15))
best_bin_cfg = bin_results.iloc[0].to_dict()
print('Best binary config:', best_bin_cfg)

In [ ]:
# Multi-class (pathological only) grouped CV search
X_patho = X_spk[y_bin_spk == 0].copy()
y_patho = y_multi_spk[y_bin_spk == 0].copy()
g_patho = groups_spk[y_bin_spk == 0].copy()

multi_models = {
    'RandomForest': rf_base,
    'XGBoost': xgb_base,
    'LightGBM': lgb_base,
    # 'Voting-Soft': VotingClassifier(estimators=estimators, voting='soft'),
    # 'Stacking': StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(multi_class='multinomial', class_weight='balanced'), cv=3, n_jobs=-1),
}

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
multi_rows = []

# Same focused experiments for multi-class
for exp in experiments:
    for model_name, model in multi_models.items():
        fold_metrics = []
        for tr, te in cv.split(X_patho, y_patho, g_patho):
            X_tr, X_te = X_patho.iloc[tr], X_patho.iloc[te]
            y_tr, y_te = y_patho.iloc[tr], y_patho.iloc[te]

            le = LabelEncoder()
            y_tr_enc = le.fit_transform(y_tr)
            y_te_enc = le.transform(y_te)

            pass_k = exp['k'] if isinstance(exp['k'], int) else 50 
            pipe = make_pipe(clone(model), sel_type=exp['sel'], k=pass_k, l1_c=exp['c'] if exp['c'] is not None else 0.05)
            pipe.fit(X_tr, y_tr_enc)
            pred_enc = pipe.predict(X_te)

            fold_metrics.append({
                'accuracy': accuracy_score(y_te_enc, pred_enc),
                'balanced_accuracy': balanced_accuracy_score(y_te_enc, pred_enc),
                'f1_macro': f1_score(y_te_enc, pred_enc, average='macro', zero_division=0),
            })

        fold_df = pd.DataFrame(fold_metrics)
        label = f"{exp['k']} ({str(exp['sel']).upper()})"
        multi_rows.append({
            'model': model_name,
            'k': label,
            'accuracy_mean': fold_df['accuracy'].mean(),
            'accuracy_std': fold_df['accuracy'].std(),
            'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
            'balanced_accuracy_std': fold_df['balanced_accuracy'].std(),
            'f1_macro_mean': fold_df['f1_macro'].mean(),
            'f1_macro_std': fold_df['f1_macro'].std(),
        })

multi_results = pd.DataFrame(multi_rows).sort_values(
    by=['balanced_accuracy_mean', 'f1_macro_mean'], ascending=False
).reset_index(drop=True)

display(multi_results.head(15))
best_multi_cfg = multi_results.iloc[0].to_dict()
print('Best multi-class config:', best_multi_cfg)

In [ ]:
# Plot top configs
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_bin_plot = bin_results.head(10).copy()
top_bin_plot['label'] = top_bin_plot['model'] + ' (k=' + top_bin_plot['k'].astype(str) + ')'
sns.barplot(data=top_bin_plot, x='f1_macro_mean', y='label', ax=axes[0], color='#4C78A8')
axes[0].set_title('Binary grouped CV — top configs by macro-F1')
axes[0].set_xlabel('mean macro-F1')

top_multi_plot = multi_results.head(10).copy()
top_multi_plot['label'] = top_multi_plot['model'] + ' (k=' + top_multi_plot['k'].astype(str) + ')'
sns.barplot(data=top_multi_plot, x='f1_macro_mean', y='label', ax=axes[1], color='#F58518')
axes[1].set_title('Multi-class grouped CV — top configs by macro-F1')
axes[1].set_xlabel('mean macro-F1')

plt.tight_layout()
plt.show()

## How to use this notebook

1. Run all cells top-to-bottom.
2. Compare v3 grouped-CV means/stdev against v2 single-split metrics.
3. Start from `best_bin_cfg` and `best_multi_cfg` for final refits / reporting.
4. If needed, tighten `K_CANDIDATES` around the best `k` and rerun for faster iteration.

In [ ]:
# Final report: best config rows + OOF metrics + confusion matrices (no file export)
from sklearn.metrics import classification_report, confusion_matrix

# --- 1) Export (in-memory) best config rows ---
best_bin_row = (
    bin_results[
        (bin_results['model'] == best_bin_cfg['model']) &
        (bin_results['k'] == best_bin_cfg['k'])
    ]
    .head(1)
    .copy()
)

best_multi_row = (
    multi_results[
        (multi_results['model'] == best_multi_cfg['model']) &
        (multi_results['k'] == best_multi_cfg['k'])
    ]
    .head(1)
    .copy()
)

print('=== BEST CONFIG ROWS (exported as DataFrames in memory) ===')
print('best_bin_row variable:')
display(best_bin_row)
print('best_multi_row variable:')
display(best_multi_row)

# --- 2) Recompute out-of-fold predictions for the BEST binary config ---
best_bin_model_name = best_bin_cfg['model']
best_bin_k = int(best_bin_cfg['k'])
best_bin_model = clone(binary_models[best_bin_model_name])

cv_bin = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

y_true_bin_all, y_pred_bin_all = [], []
fold_rows_bin = []

for fold, (tr, te) in enumerate(cv_bin.split(X_spk, y_bin_spk, groups_spk), start=1):
    X_tr, X_te = X_spk.iloc[tr], X_spk.iloc[te]
    y_tr, y_te = y_bin_spk.iloc[tr], y_bin_spk.iloc[te]

    pipe = make_pipe(clone(best_bin_model), best_bin_k)
    pipe.fit(X_tr, y_tr)

    p_tr = pipe.predict_proba(X_tr)[:, 1]
    best_thr, best_f1_train = 0.5, -1.0
    for thr in THRESHOLD_GRID:
        pred_tr = (p_tr >= thr).astype(int)
        f1_train = f1_score(y_tr, pred_tr, average='macro', zero_division=0)
        if f1_train > best_f1_train:
            best_f1_train = f1_train
            best_thr = float(thr)

    p_te = pipe.predict_proba(X_te)[:, 1]
    pred_te = (p_te >= best_thr).astype(int)

    y_true_bin_all.extend(y_te.tolist())
    y_pred_bin_all.extend(pred_te.tolist())

    fold_rows_bin.append({
        'fold': fold,
        'threshold': best_thr,
        'accuracy': accuracy_score(y_te, pred_te),
        'balanced_accuracy': balanced_accuracy_score(y_te, pred_te),
        'f1_macro': f1_score(y_te, pred_te, average='macro', zero_division=0),
    })

bin_fold_report = pd.DataFrame(fold_rows_bin)

print('\n=== BEST BINARY CONFIG: OOF METRICS ===')
print(f"model={best_bin_model_name}, k={best_bin_k}")
print((bin_fold_report.mean(numeric_only=True).round(4)).to_frame('mean').T)
print((bin_fold_report.std(numeric_only=True).round(4)).to_frame('std').T)

print('\nBinary classification report (OOF):')
print(classification_report(y_true_bin_all, y_pred_bin_all, zero_division=0))

cm_bin = confusion_matrix(y_true_bin_all, y_pred_bin_all, labels=[0, 1])
cm_bin_df = pd.DataFrame(cm_bin, index=['true_0', 'true_1'], columns=['pred_0', 'pred_1'])
print('Binary confusion matrix (OOF):')
display(cm_bin_df)

# --- 3) Recompute out-of-fold predictions for the BEST multi-class config (pathological only) ---
best_multi_model_name = best_multi_cfg['model']
best_multi_k = int(best_multi_cfg['k'])
best_multi_model = clone(multi_models[best_multi_model_name])

cv_multi = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

y_true_multi_all, y_pred_multi_all = [], []
fold_rows_multi = []

for fold, (tr, te) in enumerate(cv_multi.split(X_patho, y_patho, g_patho), start=1):
    X_tr, X_te = X_patho.iloc[tr], X_patho.iloc[te]
    y_tr, y_te = y_patho.iloc[tr], y_patho.iloc[te]

    le = LabelEncoder()
    y_tr_enc = le.fit_transform(y_tr)

    pipe = make_pipe(clone(best_multi_model), best_multi_k)
    pipe.fit(X_tr, y_tr_enc)

    pred_enc = pipe.predict(X_te)
    pred_lbl = le.inverse_transform(pred_enc)

    y_true_multi_all.extend(y_te.astype(str).tolist())
    y_pred_multi_all.extend(pd.Series(pred_lbl).astype(str).tolist())

    fold_rows_multi.append({
        'fold': fold,
        'accuracy': accuracy_score(y_te, pred_lbl),
        'balanced_accuracy': balanced_accuracy_score(y_te, pred_lbl),
        'f1_macro': f1_score(y_te, pred_lbl, average='macro', zero_division=0),
    })

multi_fold_report = pd.DataFrame(fold_rows_multi)

print('\n=== BEST MULTI-CLASS CONFIG: OOF METRICS ===')
print(f"model={best_multi_model_name}, k={best_multi_k}")
print((multi_fold_report.mean(numeric_only=True).round(4)).to_frame('mean').T)
print((multi_fold_report.std(numeric_only=True).round(4)).to_frame('std').T)

print('\nMulti-class classification report (OOF):')
print(classification_report(y_true_multi_all, y_pred_multi_all, zero_division=0))

multi_labels = sorted(pd.Series(y_true_multi_all).unique())
cm_multi = confusion_matrix(y_true_multi_all, y_pred_multi_all, labels=multi_labels)
cm_multi_df = pd.DataFrame(
    cm_multi,
    index=[f'true_{x}' for x in multi_labels],
    columns=[f'pred_{x}' for x in multi_labels],
)
print('Multi-class confusion matrix (OOF):')
display(cm_multi_df)

# --- 4) Plot confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm_bin_df,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0],
)
axes[0].set_title('Binary OOF Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(
    cm_multi_df,
    annot=True,
    fmt='d',
    cmap='Oranges',
    ax=axes[1],
)
axes[1].set_title('Multi-class OOF Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

In [ ]:
# Debug: why Functional group may be missing
if 'pathology_de' in df.columns:
    raw_counts = df['pathology_de'].astype(str).str.strip().value_counts()
    print('Top raw pathology_de counts:')
    display(raw_counts.head(30).to_frame('count'))

    map_keys = set(DISEASE_GROUP_MAP.keys())
    raw_labels = set(df['pathology_de'].astype(str).str.strip().unique())
    missing_map = sorted(raw_labels - map_keys)

    print('\nRaw labels that are NOT in DISEASE_GROUP_MAP (first 40):')
    print(missing_map[:40])

    functional_keys = [k for k, v in DISEASE_GROUP_MAP.items() if v == 'Functional']
    print('\nFunctional keys in map:', functional_keys)
    print('Counts for Functional keys present in data:')
    rows = []
    for k in functional_keys:
        rows.append({'label': k, 'count': int(raw_counts.get(k, 0))})
    display(pd.DataFrame(rows).sort_values('count', ascending=False))

print('\nCurrent grouped target counts:')
display(df['target_label'].value_counts().to_frame('count'))